# Lab 5.2 · Pseudoinverse and Orthogonal Projections

Build the pseudoinverse from positive singular values and distinguish its input and output projectors.

Wooseok Ha, MAS110 Lecture 5, printed slides 49–59, especially equations (5.18), (5.19), and (5.22).

Lecture 4 found the closest reachable output. Lecture 5 shows that a pseudoinverse can produce it for any matrix. This rectangular example gives the input and output projectors different dimensions so their jobs are visible.

Run the cells in order with NumPy. The code and notation follow the web lab.

## Notation

- $A^+=VD^{-1}U^{\top}$ · Invert only the $r$ positive singular values in the compact SVD. The pseudoinverse maps from $\mathbb{R}^n$ to $\mathbb{R}^d$.

- $AA^+=UU^{\top}$ · An $n\times n$ projector onto $\operatorname{Col}(A)$ in output space.

- $A^+A=VV^{\top}$ · A $d\times d$ projector onto the row space in input space. Its complementary subspace is $\operatorname{Null}(A)$.

- $\hat{x}$ and $\hat{b}$ · $\hat{x}=A^+b$ gives a least-squares input and $\hat{b}=A\hat{x}$ gives the unique fitted output.

## 1. Invert the two nonzero scales

The duplicate columns give one zero singular value in exact arithmetic. For this example a cutoff of $10^{-12}$ separates it from the two positive scales.

**Predict first.** How many rows does the pseudoinverse have?

In [ ]:
import numpy as np
A = np.array([[1., 1., 0.], [1., 1., 0.], [0., 0., 1.], [0., 0., 0.]])
b = np.array([1., 3., 2., 4.])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
keep = s > 1e-12
A_plus = (Vt[keep, :].T / s[keep]) @ U[:, keep].T
print("Singular values =", s, "\nKept =", keep, "\nA_plus =", A_plus)

The pseudoinverse has three rows and four columns.

$$A^+=VD^{-1}U^{\top}$$

## 2. Apply the output projector

The first two output entries become their average. The third is unchanged and the fourth drops out.

**Predict first.** What happens to the fourth entry of $b$ in the residual?

In [ ]:
P_col = A @ A_plus
b_hat = P_col @ b
residual = b - b_hat
print("Output projector =", P_col, "\nFit =", b_hat, "\nResidual =", residual)

The first two residual entries cancel against either of the repeated columns.

$$A^{\top}(b-\hat{b})=0$$

## 3. Compare two least-squares inputs

The row-space projector removes the null-space shift from the alternative input.

**Predict first.** Which input has smaller squared length?

In [ ]:
P_row = A_plus @ A
x_hat = A_plus @ b
x_other = x_hat + np.array([1., -1., 0.])
x_projected = P_row @ x_other
print("Input projector =", P_row, "\nShortest input =", x_hat, "\nShifted input =", x_other, "\nProjected input =", x_projected)

The two squared input lengths are $6$ and $8$. Both inputs map to the same fit.

$$A\hat{x}=Ax_{\mathrm{other}}=\hat{b}$$

## Two projectors, two spaces

### Cancel the compact factors in the right order

Using $V^{\top}V=I_r$ and $DD^{-1}=I_r$ gives $AA^+=UU^{\top}$. Canceling $U^{\top}U$ instead gives $A^+A=VV^{\top}$. Both products are symmetric and idempotent. Together with $AA^+A=A$ and $A^+AA^+=A^+$, their symmetry gives the Penrose conditions.

### Project first, then choose an input

The residual $b-\hat{b}$ is orthogonal to the column space. Every minimizer has the same fitted output and can be written as $\hat{x}+z$ with $z\in\operatorname{Null}(A)$. The vector $\hat{x}$ lies in the row space, perpendicular to $z$. Pythagoras therefore makes it the unique minimizer of input length among all least-squares solutions.

$$\lVert\hat{x}+z\rVert_2^2=\lVert\hat{x}\rVert_2^2+\lVert z\rVert_2^2$$

## Verify the example

Run these checks on the example before changing the input.

In [ ]:
assert A_plus.shape == (3, 4)
assert np.allclose(A_plus, np.linalg.pinv(A))
assert np.allclose(P_col.T, P_col) and np.allclose(P_row.T, P_row)
assert np.allclose(P_col @ A, A) and np.allclose(P_row @ A_plus, A_plus)
assert np.allclose(P_col @ P_col, P_col) and np.allclose(P_row @ P_row, P_row)
assert np.allclose(b_hat, [2, 2, 2, 0])
assert np.allclose(residual, [-1, 1, 0, 4])
assert np.allclose(A.T @ residual, 0)
assert np.allclose(x_hat, [1, 1, 2]) and np.allclose(x_projected, x_hat)
assert np.allclose(A @ x_other, b_hat)
assert np.allclose([x_hat @ x_hat, x_other @ x_other], [6, 8])

## Check your understanding

Which projector acts on the observed vector $b$?

$AA^+$, because $b$ has $n$ entries. Its image is the space of outputs the matrix can reach.

Can adding $(1,-1,0)^{\top}$ to $\hat{x}$ improve the fit?

It gives exactly the same fit. It lies in the null space and increases the squared input length by $2$.

## References

- [NumPy pseudoinverse](https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html)
- [Original Chapter 5.2 notebook](https://github.com/kyunghyuncho/Foundations_of_LADS/blob/main/Ch5-2%20SVD%20and%20Projections.ipynb)